# Find samples using Advanced Search and send them to Flexible Intel Feed

This notebook showcases how to search for relevant samples using Advanced Search and send them to Flexible Intel Feed.
A flow like this one gives us a simple yet useful way to utilize our custom criteria for selectively sending samples to Flexible Intel Feed analysis.
To make this workflow send samples to Flexible Intel Feed, we need to have Flexible Intel Feed enabled on our Spectra Analyze instance.

### Used Spectra Analyze functions
- **advanced_search_v3_aggregated**
- **reanalyze_samples_v2**

### Credentials
Credentials are loaded from a local file instead of being written here in plain text.
To learn how to create credentials file, see the **Storing and using credentials** section in the [README file](./README.md)


### 1. Search for samples using Advanced Search
First, we have to do the required Python imports and load our credentials. After that, we create our A1000 object.

In [ ]:
import json
from ReversingLabs.SDK.a1000 import A1000

CREDENTIALS = json.load(open('credentials.json'))
HOST = CREDENTIALS.get("a1000").get("a1000_url")
TOKEN = CREDENTIALS.get("a1000").get("token")
USER_AGENT = json.load(open('../user_agent.json'))["user_agent"]

a1000_obj = A1000(
    host=HOST,
    token=TOKEN,
    user_agent=USER_AGENT,
    verify=False
)

Now we can perform Advanced Search to find sample hashes using our search criteria.
Change the `SEARCH_QUERY` variable to fit your sample search needs. The given example searches for Android malware not older than 3 days.

In [ ]:
SEARCH_QUERY = "classification:malicious type:android firstseen:3d-"

search_resp = a1000_obj.advanced_search_v3_aggregated(
    query_string=SEARCH_QUERY
)

print(search_resp)

This action gave us a list of Python dicts. Now we can iterate through that list and request sample reanalysis for each entry.

### 2. Reanalyze samples to send them to Flexible Intel Feed
Make sure to set `titanium_cloud` to `True` as uploading the sample to the cloud is required for triggering Flexible Intel Feed analysis.

In [ ]:
sha1_list = []

for sample in search_resp:
    sha1_list.append(sample.get("sha1"))

reanalyze_resp = a1000_obj.reanalyze_samples_v2(
    hash_input=sha1_list,
    titanium_cloud=True
)
print(reanalyze_resp.text)